In [1]:
import time
import re
import numpy as np
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
import pprint
import logging
import pickle

import data

In [2]:
class ColorHandler(logging.StreamHandler):
    GRAY8 = "38;5;245"
    GRAY7 = "38;5;247"
    ORANGE = "33"
    RED = "31;5;203"
    BRIGHT_RED = "1;31"
    WHITE = 0

    def emit(self, record):
        level_color_map = {
            logging.DEBUG: self.GRAY8,
            logging.INFO: self.GRAY7,
            logging.WARNING: self.ORANGE,
            logging.ERROR: self.RED,
            logging.CRITICAL: self.BRIGHT_RED,
        }

        csi = f"{chr(27)}["  # control sequence introducer
        color = level_color_map.get(record.levelno, self.WHITE)
        print(f"{csi}{color}m{record.msg}{csi}m")


logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")

console_handler = ColorHandler()
# console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)
console_format = logging.Formatter('%(name)s - %(levelname)s - %(message)s')
console_handler.setFormatter(console_format)

file_handler = logging.FileHandler('tcg.log')
file_handler.setLevel(logging.DEBUG)
file_format = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
file_handler.setFormatter(file_format)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

logger.debug("This is a debug message")
logger.info("This is an info message")
logger.warning("This is a warning")
logger.error("This is an error")
logger.critical("This is critical")

This is a debug message
This is an info message
This is a warning
This is an error
This is critical


In [3]:
class Vendor:
    __lastId = 0

    def __init__(self, name, url, shipping, reduced_shipping=0):
        self.name = name
        self.url = url
        self.shipping = shipping
        self.reduced_shipping = reduced_shipping
        self.id = Vendor.__lastId
        Vendor.__lastId += 1


class Listing:
    def __init__(self, name, url, price, vendor=None):
        self.name = name
        self.url = url
        self.price = price
        self.vendor = vendor

def url_build(tcd_id):
    url = "https://www.tcgplayer.com/product/{}?".format(tcd_id)
    url += "&Language=English"
    url += "&Condition=Near+Mint|Lightly+Played"
    return url

In [4]:
# Get card processed card data from scryfall
card_data = data.init_cards()
with open("b_deck.txt", encoding="utf8") as f:
    lines = f.readlines()
card_names = []

# strip amounts of cards, We assumed only 1 of each card
for line in lines:
    card_names.append(line[1:].strip())
cards = {}

for card_name in card_names:
    try:
        cards[card_name] = card_data[card_name]
    except KeyError as e:
        pprint.pprint(e)
        print("failed to find card" + card_name)
        exit(1)

card_listings = {}
vendors = {}

In [5]:
# driver init
driver = webdriver.Firefox()
driver.implicitly_wait(3)
url = url_build(196523)
driver.get(url)
time.sleep(1.5)
driver.find_elements(By.CLASS_NAME, "tcg-input-select__trigger-container")[1].click()
time.sleep(0.5)
drop_downs = driver.find_elements(By.CLASS_NAME, "tcg-base-dropdown__item-content")
for drop_down in drop_downs:
    if drop_down.text == "50":
        drop_down.click()

In [6]:
def read_card(card, vendors, driver):
    prices = []
    for printing in card:
        prices.append(printing["price"])
    price_min, price_max = min(prices), max(prices)
    if price_min > 10:
        price_cutoff = price_min * 1.3 + 1
    else:
        price_cutoff = max(price_min * 1.3, price_min + 0.3)
    index_to_delete = []
    for idx, price in enumerate(prices):
        if price > price_cutoff:
            index_to_delete.append(idx)
    index_to_delete.sort(reverse=True)
    for idx in index_to_delete:
        card.pop(idx)
    if len(card) > 15:
        card = card[:15]
    listings = []

    for printing in card:
        url = url_build(printing["tcgplayer_id"])
        driver.get(url)
        time.sleep(0.5)

        elements = driver.find_elements(By.CLASS_NAME, "listing-item")

        for element in elements:

            price_element = element.find_element(By.CLASS_NAME, "listing-item__listing-data__info")

            match = re.search(r"Over \$\d+", price_element.text)
            shipping_type = None
            if match:
                shipping_type = int(match.group().split(" ")[1][1:])
            shipping = 0
            match = re.search(r"\$\d+\.\d{2}\sShipping[^:]", price_element.text + " ")
            if match:
                shipping = float(match.group().split(" ")[0][1:])
            price = float(
                price_element.find_element(By.CLASS_NAME, "listing-item__listing-data__info__price").text[1:].replace(
                    ',', ''))
            if not shipping_type:
                price += shipping
                shipping_type = 0
            elif shipping_type == 50:
                continue

            vendor = element.find_element(By.CLASS_NAME, "seller-info__name")
            vendor_name = vendor.text
            vendor_url = vendor.get_attribute("href")
            if vendor_name not in vendors:
                vendors[vendor_name] = Vendor(vendor_name, vendor_url, shipping)
                print("adding new vendor:{}".format(vendor_name))
            # print(vendor_name, vendor_url, price, shipping, shipping_type)
            listing = Listing(printing["name"], url, price, vendors[vendor_name])
            listings.append(listing)

    return listings

for card in cards:
    card_listings[card] = read_card(cards[card], vendors, driver)

with open("card_read", "wb") as f:
    pickle.dump([card_listings, vendors], f)

adding new vendor:BaramoreTCG
adding new vendor:Bulba Bulk TCG
adding new vendor:Panda Trading Post
adding new vendor:LuxuryCollectorGuild
adding new vendor:MAD Gaming Central
adding new vendor:Warren Wonders
adding new vendor:Eclectic Enormous
adding new vendor:QtNegahi
adding new vendor:AndyJackMtg
adding new vendor:AmpedMTG
adding new vendor:Legit MTG
adding new vendor:ScavengerGroundGames
adding new vendor:Purple Lotus Bazaar
adding new vendor:ShrugThisShop
adding new vendor:Turtle Nation
adding new vendor:GGAZ_CCG
adding new vendor:EastWinds Traders
adding new vendor:GameDay Box
adding new vendor:CollectorsTreasureBx
adding new vendor:Ben TCG Collectables
adding new vendor:Kanewnews Cards
adding new vendor:Frosthold Games
adding new vendor:Card & Board Co
adding new vendor:Cardline
adding new vendor:Gnome Nation Games
adding new vendor:KozmicKaijus cards
adding new vendor:Lunar Hobbies
adding new vendor:Everything Geek
adding new vendor:PersonalMagic ToGo
adding new vendor:CatCard

In [11]:
class State:
    def __init__(self, vendor_sales, listings, price=0):
        self.vendor_sales = vendor_sales
        self.listings = listings
        self.price = price

    def add_card(self, listing):
        vendor_sales = np.copy(self.vendor_sales)
        listings = self.listings.copy()
        listings.append(listing)

        vendor = listing.vendor
        price = self.price + listing.price

        if not vendor_sales[vendor.id] == -1:
            if vendor_sales[vendor.id] == 0:
                price += vendor.shipping

            vendor_sales[vendor.id] += listing.price

            if vendor_sales[vendor.id] >= 5:
                price = price - vendor.shipping + vendor.reduced_shipping
                vendor_sales[vendor.id] = -1

        return State(vendor_sales, listings, price)
#BFS going card by card.
def optimize(vendors, card_listings):
    run_untested = False
    if run_untested:
        vendor_score = np.zeros(len(vendors), dtype=np.int32)
        for card in card_listings:
            for listing in card_listings[card]:
                vendor_score[listing.vendor.id] += 1
        #remove listings that only have vendors_count of 1 and are not the cheapest
        to_remove = []
        for listings in card_listings.values():
            listings.sort(key=lambda x: x.price)
            for i in reversed(range(len(listings))[1:]):
                if vendor_score[listings[i].vendor.id] == 1:
                    listings.pop(i)
                    to_remove.append(listings[i].vendor.name)
        for v in to_remove:
            vendors.pop(v)
        #re-id vendors after removing some of them
        for i, vendor in enumerate(vendors.values()):
            vendor.id = i
            Vendor.__lastId = i + 1

    vendor_score = np.zeros(len(vendors), dtype=np.int32)
    for card in card_listings:
        for listing in card_listings[card]:
            vendor_score[listing.vendor.id] += 1
    scaled_vendor_score = [(x * 2) ** (2/3) * 0.01 for x in vendor_score]

    for listings in card_listings.values():
        listings.sort(key=lambda x: x.price - scaled_vendor_score[x.vendor.id])

    #prev_time = 0
    #prev_size = 1
    options = [State(np.zeros(len(vendors)), [])]
    for listings in sorted(card_listings.values(), key=lambda x: x[0].price, reverse=True):
        start = time.time()
        new_options = []
        for option in options:
            shipping_covered = 1
            shipping_not_covered = 2
            fresh_vendor = 2
            for listing in listings:
                price = listing.price
                vendor = listing.vendor
                vendor_sales = option.vendor_sales[vendor.id]
                if shipping_covered and (price >= 5 or vendor_sales == -1):
                    shipping_covered = 0
                    new_options.append(option.add_card(listing))
                elif shipping_not_covered and vendor_sales > 0:
                    shipping_not_covered -= 1
                    new_options.append(option.add_card(listing))
                elif fresh_vendor and vendor_sales == 0:
                    fresh_vendor -= 1
                    new_options.append(option.add_card(listing))
                elif shipping_covered + shipping_not_covered + fresh_vendor == 0:
                    break

        time_taken = time.time() - start
        print("time taken: {} Option size: {}".format(time_taken, len(new_options)))
        cull_time = 0.01
        if time_taken > cull_time:
            print("Culling")
            new_options.sort(key=lambda o: o.price)
            #scale = cull_time / prev_time
            #new_options = new_options[:int(prev_size * scale)]
            new_options = new_options[:len(new_options) // 10]
        #prev_time = time_taken
        #prev_size = len(new_options)
        options = new_options
    options.sort(key=lambda o: o.price)
    print(options[0].price)
    for listing in options[0].listings:
        print("card: {}, price: {}, vendor: {} ,link {}".format(listing.name, listing.price, listing.vendor.name,
                                                                listing.url, ))
    return options[0]


with open("card_read", "rb") as f:
    card_listings, vendors = pickle.load(f)

option = optimize(vendors, card_listings)

time taken: 7.009506225585938e-05 Option size: 3
time taken: 0.0001277923583984375 Option size: 7
time taken: 0.0001246929168701172 Option size: 14
time taken: 0.000141143798828125 Option size: 28
time taken: 0.0008058547973632812 Option size: 84
8.19
card: Arcane Denial, price: 1.24, vendor: JLMagic ,link https://www.tcgplayer.com/product/203658?&Language=English&Condition=Near+Mint|Lightly+Played
card: An Offer You Can't Refuse, price: 0.7, vendor: ClayWarForge ,link https://www.tcgplayer.com/product/590828?&Language=English&Condition=Near+Mint|Lightly+Played
card: Archmage Emeritus, price: 0.67, vendor: ManaNest ,link https://www.tcgplayer.com/product/624158?&Language=English&Condition=Near+Mint|Lightly+Played
card: Anhelo, the Painter, price: 0.22, vendor: 6AGames Direct ,link https://www.tcgplayer.com/product/268380?&Language=English&Condition=Near+Mint|Lightly+Played
card: Baral and Kari Zev, price: 0.12, vendor: ManaNest ,link https://www.tcgplayer.com/product/624189?&Language=E

In [ ]:
#only puts cards in cart
def buy(option, driver):
    for listing in option.listings:
        driver.get(listing.url)
        time.sleep(0.5)
        elements = driver.find_elements(By.CLASS_NAME, "listing-item")

        for element in elements:
            price_element = element.find_element(By.CLASS_NAME, "listing-item__listing-data__info")

            match = re.search(r"Over \$\d+", price_element.text)
            shipping_type = None
            if match:
                shipping_type = int(match.group().split(" ")[1][1:])
            if shipping_type == 50:
                continue

            vendor = element.find_element(By.CLASS_NAME, "seller-info__name")
            vendor_name = vendor.text
            if vendor_name == listing.vendor.name:
                add_element = element.find_element(By.CLASS_NAME, "add-to-cart")
                print("found listing")
                button = add_element.find_element(By.TAG_NAME, "button")
                header = driver.find_element(By.CLASS_NAME, "horizontal-filters-bar")
                driver.execute_script("""var element = arguments[0];element.parentNode.removeChild(element); """,
                                      header)
                driver.execute_script("arguments[0].scrollIntoView();", button)
                # time.sleep(1)
                ActionChains(driver).scroll_to_element(button).perform()
                print(button.text)
                print(button.get_attribute('innerHTML'))
                print(button)

                # return

                button.click()
                driver.find_element(By.CLASS_NAME, "tcg-snackbar__message")
                # return

                break

buy(option, driver)

In [ ]:
import math
def optimize(vendors, card_listings):
    v = set()
    v2 = []
    for vendor in vendors:
        #print(vendors[vendor].id)
        #vendors[vendor].id -= 961
        v.add(vendors[vendor].id)
        v2.append(vendors[vendor].id)
    v2.sort()
    print(v2[0:100])
    print(len(vendors))
    print(len(v))
    #return
    temp =[]
    for x in range (len(vendors)):
        temp.append([])
    #print([[] for x in range(len(vendors))])
    options = [State([0] * len(vendors), [])]
    vendor_score = [0] * len(vendors)
    #print(card_listings)
    for card in card_listings:
        for listing in card_listings[card]:
            #print(listing)
            #print(listing.vendor.id)
            vendor_score[listing.vendor.id] += 1
    print(vendor_score)
    vendor_score = [x * 0.02 for x in vendor_score]
    for card in card_listings:
        card_listings[card].sort(key=lambda x: x.price - vendor_score[x.vendor.id])
    w = 0
    #card_listings.pop("Dark Ritual")
    #card_listings.pop("Timeline Culler")
    for x in card_listings.keys():
        y = card_listings[x]
        print(x)
        print(len(y))
        c = y[0]
        p = c.price
    keys = sorted(card_listings.keys(), key=lambda x: card_listings[x][0].price, reverse=True)
    for card in keys:

        if w == 5:
            pass
            #break
        w +=1
        new_options = []
        start = time.time()
        for option in options:
            cheapest_shipping_covered = False
            under_five = 2
            if w > 50:
                fresh  = 1
            else:
                fresh = 2
            for listing in card_listings[card]:
                price = listing.price
                vendor = listing.vendor
                #if price > 5 or option.shipping_covered[vendor.id]:
                if price > 5 or vendor.id in option.shipping_covered:
                    if not cheapest_shipping_covered:
                        new_options.append(option.add_card(listing, vendor, price))
                        cheapest_shipping_covered = True

                elif option.under_five[vendor.id] and under_five:
                    under_five -= 1
                    new_options.append(option.add_card(listing, vendor, price))
                elif fresh:
                    new_options.append(option.add_card(listing, vendor, price))
                    fresh -= 1
                elif not cheapest_shipping_covered and not under_five and not fresh:
                    break
        options = new_options
        end = time.time()
        print("time taken: {} Option size: {}".format(end - start, len(options)))
        if end - start > 0.5:
            options.sort(key=lambda o: o.price)
            options = options[:len(options) // 10]
            # cull
    options.sort(key=lambda o: o.price)
    print(options[0].price)
    print(options[0].card_to_store)
    for listing in options[0].card_to_store:
        print("card: {}, price: {}, vendor: {} ,link {}".format(listing.name, listing.price, listing.vendor.name, listing.url,))
            #pprint.pprint(array)
    return options[0]
option = optimize(vendors, card_listings)

In [ ]:
#manually go to cart. run this to see what cards are missing if any
print(len(option.listings))
missing = []
elements = driver.find_elements(By.CLASS_NAME, "name")
b = []
x = "This is a sentence. (once a day) [twice a day]"
for e in elements:
    b.append(re.sub("[\(\[].*?[\)\]]", "", e.text).strip())

for listing in option.listings:
    if listing.name not in b:
        print(listing.name)